In [25]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import openai



In [16]:
df = pd.read_json("../data/sample_dataset.jsonl", lines=True)
df_sample = df.sample(50, random_state=42)
data_to_embed = df_sample[["description", "image", "rating_number", "price", "average_rating", "parent_asin"]].to_dict(orient="records")

print(f"Loaded {len(data_to_embed)} items to embed.")


Loaded 50 items to embed.


In [17]:
data_to_embed

[{'description': 'Sensationnel Dashly lace front synthetic wigs have a wide hand tied swiss lace parting area with HD transparent lace and baby hair. Dashly wigs are prestyled yet customizable and easy to use.',
  'image': 'https://m.media-amazon.com/images/I/71APnrecqGL._SL1500_.jpg',
  'rating_number': 782,
  'price': 28.98,
  'average_rating': 4.0,
  'parent_asin': 'B07ZHP2WJX'},
 {'description': 'Non Magnetic Eyeliner and Eyelashes Kit, Magic Self Adhesive False Eyelashes and Eyeliner, 5 Pairs 5D Reusable False Lashes with No Glue, Waterproof Lash Boxes with Mirror & Tweezers',
  'image': 'https://m.media-amazon.com/images/I/71p3QWCHd-S._SL1500_.jpg',
  'rating_number': 21,
  'price': 2.99,
  'average_rating': 3.5,
  'parent_asin': 'B08XXQFF1Y'},
 {'description': "If you're in need of a midday refresher, spray Invictus on yourself to liven up your day. This men's fragrance reveals a number of notes, including hints of grapefruit, Hedione jasmine, patchouli, bay leaves, and oak moss

## Initializing The Embedding Function

In [36]:
from openai import OpenAI
import os

# Rename this to openai_client
openai_client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.environ.get("OPENROUTER_API_KEY"),
)

response = openai_client.embeddings.create(
    model="openai/text-embedding-3-small",
    input="Random text"
)


In [37]:
len(response.data[0].embedding)

1536

In [38]:
def get_embedding(text, model="openai/text-embedding-3-small"):
    response = openai_client.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding


In [39]:
get_embedding("hello")

[0.016754150390625,
 -0.055755615234375,
 0.005634307861328125,
 0.06622314453125,
 0.008941650390625,
 -0.04730224609375,
 -0.02813720703125,
 0.06109619140625,
 -0.0021839141845703125,
 -0.043731689453125,
 0.009857177734375,
 -0.033203125,
 -0.01219940185546875,
 -0.028839111328125,
 0.01462554931640625,
 0.058258056640625,
 -0.070068359375,
 0.037567138671875,
 0.01006317138671875,
 0.040802001953125,
 0.06640625,
 0.0101165771484375,
 -0.005603790283203125,
 0.0236663818359375,
 0.021759033203125,
 0.01189422607421875,
 -0.00478363037109375,
 0.0192413330078125,
 0.0243682861328125,
 -0.06488037109375,
 0.03326416015625,
 -0.044586181640625,
 0.0203857421875,
 -0.004970550537109375,
 0.01082611083984375,
 -0.01480865478515625,
 -0.026275634765625,
 0.03466796875,
 0.00234222412109375,
 -0.03326416015625,
 -0.022125244140625,
 -0.00853729248046875,
 0.05401611328125,
 0.02435302734375,
 -5.304813385009766e-06,
 0.0156097412109375,
 -0.020965576171875,
 -0.004283905029296875,
 0.032

In [41]:
client = QdrantClient(host="localhost", port=6333)

client.create_collection(
    collection_name="products",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)





True

## Testing Point Creation With Qdrant

In [42]:
# Test PointStruct insertion
test_text = "This is a test product description."
test_embedding = get_embedding(test_text)



In [43]:
point = PointStruct(
    id=1,
    vector=test_embedding,
    payload={"description": test_text}
)

### Writing embed Data to qdrant

In [44]:
pointstructs = []
for i, data in enumerate(data_to_embed):
    # Generate the embedding using the description
    embedding = get_embedding(data["description"])
    
    # Create the PointStruct and add it to our list
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data,
        )
    )

In [46]:
len(pointstructs)

50

In [ ]:
from asyncio import wait
# Push all the generated points to our Qdrant collection
client.upsert(
    collection_name="products",
    points=pointstructs,
    wait=True
)
print(f"Successfully uploaded {len(pointstructs)} products to Qdrant!")

Successfully uploaded 50 products to Qdrant!


In [50]:
def retrieve_products(query: str, limit: int = 3):
    """
    Search for products based on a natural language query.
    """
    # 1. Convert the user's text query into a vector embedding
    query_vector = get_embedding(query)
    
    # 2. Search the Qdrant database for the closest matching vectors
    search_results = client.query_points(
        collection_name="products",
        query=query_vector,
        limit=limit
    )

    return search_results

In [52]:
retrieve_products("I need some bright lipstick",limit=10).points

[ScoredPoint(id=10, version=1, score=0.47508836, payload={'description': 'Thrive Cosmetics Lip liner and plumper. This is a full size product with no box. The shade is VALISIA a bright pomegranate shade. It s absolutely gorgeous Wear it alone or pair it with your favorite lipstick or gloss. Paraben free cruelty free and vegan.', 'image': 'https://m.media-amazon.com/images/I/51WenLxxbvL._SL1486_.jpg', 'rating_number': 18, 'price': 12.0, 'average_rating': 4.0, 'parent_asin': 'B08TVG4KW7'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=19, version=1, score=0.40872014, payload={'description': "Excellent after-sales service:We're confident about the quality of our FANICEA 4 pcs matte lipstick set A. If you have any question about our product or not satisfied with our product. Please feel free to contact us. We will try our best to solve your problems. Our most important mission is to satisfy customers and shop happily!Size: 11.6*2.4*1.3cm(each count)Package Included: 4 cou